# B.2.4 — Multi-feature ablation at L20 of Gemma 2 9B

**Goal:** **Quantitatively** test the representational redundancy hypothesis. Instead of ablating a single feature (B.2.3 showed a null effect), ablate **the top-K most gender-discriminative features** simultaneously, with K ∈ {0, 1, 2, 3, 5, 10, 20, 50}, and measure how benchmark accuracy drops with K.

**Hypothesis prediction:**
- **If redundancy is real:** accuracy degrades gradually with K (several features are needed to break the signal).
- **If single-feature sufficed:** it would already drop a lot at K=1 (which we did not see in B.2.3).

**Setup:** L20 only (focusing on the case where the top-1 feature WAS active in 100% of feminine items at the target token — sanity check confirmed). 228 items × 8 K values = 1824 forward passes.

**Output:** `gemma9b_multifeature_ablation_L20.json` + table K vs Δfem vs Fisher p.

**Estimated runtime:** ~12-15 min on Colab L4.

## 1. Setup

In [ ]:
!pip install -q sae-lens transformer-lens scipy

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

device = "cuda"
DTYPE = torch.bfloat16
LAYER = 20
MODEL_NAME = "gemma-2-9b"
HF_MODEL_ID = f"google/{MODEL_NAME}"
SAE_RELEASE = "gemma-scope-9b-pt-res-canonical"

K_VALUES = [0, 1, 2, 3, 5, 10, 20, 50]

torch.cuda.empty_cache(); gc.collect()

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL_ID, torch_dtype=DTYPE, device_map={"": device},
    low_cpu_mem_usage=True, attn_implementation="eager",
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

print(f"Loading SAE layer_{LAYER}/width_16k/canonical...")
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=f"layer_{LAYER}/width_16k/canonical", device=device)
if isinstance(sae, tuple): sae = sae[0]
sae = sae.to(DTYPE)
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

## 2. Top-K features de gender em L20 (do B.2.2)

In [ ]:
import json

with open("gemma9b_probes_layer20_31.json") as f:
    probes = json.load(f)

top_features = probes["probe_results"]["gender"][str(LAYER)]["top_features"]
print(f"Top features loaded: {len(top_features)}")
print("Top-20 por |effect_size|:")
print(f"  {'rank':>4} {'feat_id':>8} {'effect_d':>10} {'mean_pos':>10} {'mean_neg':>10}  bias")

ranked = sorted(top_features, key=lambda f: abs(f["effect_size"]), reverse=True)
for i, f in enumerate(ranked[:20]):
    bias = "F" if f["effect_size"] > 0 else "M"
    print(f"  {i+1:>4} {f['feature_id']:>8} {f['effect_size']:>+10.3f} {f['mean_pos']:>10.3f} {f['mean_neg']:>10.3f}  {bias}")

TOP_FEATURE_IDS = [f["feature_id"] for f in ranked]
print(f"\nFull list ready (n={len(TOP_FEATURE_IDS)}).")

## 3. Hook de multi-feature ablation

In [ ]:
from contextlib import contextmanager

def make_multi_ablation_hook(feature_ids):
    if not feature_ids:
        return None
    fids = torch.tensor(list(feature_ids), dtype=torch.long, device=device)
    def hook(module, inputs, output):
        if isinstance(output, tuple):
            h, rest = output[0], output[1:]
        else:
            h, rest = output, None
        with torch.no_grad():
            feat = sae.encode(h)
            modified = feat.clone()
            modified[..., fids] = 0.0
            recon_orig = sae.decode(feat)
            recon_mod = sae.decode(modified)
            new_h = h - recon_orig + recon_mod
        if rest is not None:
            return (new_h,) + rest
        return new_h
    return hook

@contextmanager
def ablate_multi(feature_ids):
    hook = make_multi_ablation_hook(feature_ids)
    if hook is None:
        yield; return
    handle = model.model.layers[LAYER].register_forward_hook(hook)
    try:
        yield
    finally:
        handle.remove()

test_text = "A menina bonita foi à"
test_ids = tokenizer(test_text, return_tensors="pt").input_ids.to(device)
with torch.no_grad():
    logits_b = model(test_ids).logits
with ablate_multi(TOP_FEATURE_IDS[:5]):
    with torch.no_grad():
        logits_a = model(test_ids).logits
diff = (logits_b.float() - logits_a.float()).abs().max().item()
print(f"Sanity: ablating 5 features changes logits (max |Δ|={diff:.4f})")

## 4. Gender benchmark (identical to B.2.3)

In [ ]:
import random

ADJECTIVES = [
    ("bonito", "bonita"), ("alto", "alta"), ("baixo", "baixa"),
    ("gordo", "gorda"), ("magro", "magra"), ("velho", "velha"),
    ("novo", "nova"), ("feio", "feia"), ("rico", "rica"),
    ("pobre", "pobre"), ("cansado", "cansada"), ("animado", "animada"),
    ("dedicado", "dedicada"), ("organizado", "organizada"),
    ("preocupado", "preocupada"), ("calmo", "calma"),
    ("nervoso", "nervosa"), ("famoso", "famosa"),
    ("talentoso", "talentosa"), ("corajoso", "corajosa"),
    ("preguiçoso", "preguiçosa"), ("cuidadoso", "cuidadosa"),
    ("carinhoso", "carinhosa"), ("generoso", "generosa"),
    ("sério", "séria"), ("simpático", "simpática"),
    ("antipático", "antipática"), ("tímido", "tímida"),
    ("atrevido", "atrevida"), ("educado", "educada"),
    ("esperto", "esperta"), ("lento", "lenta"),
    ("rápido", "rápida"), ("quieto", "quieta"),
    ("barulhento", "barulhenta"), ("chato", "chata"),
    ("divertido", "divertida"), ("salgado", "salgada"),
    ("amargo", "amarga"), ("doce", "doce"),
]
SUBJECTS_FEM = ["A menina","A professora","A médica","A engenheira","A advogada","A diretora","A aluna","A cantora","A cozinheira","A vizinha","A mãe","A filha","A irmã","A avó","A tia","A enfermeira","A motorista","A secretária","A escritora","A atriz"]
SUBJECTS_MASC = ["O menino","O professor","O médico","O engenheiro","O advogado","O diretor","O aluno","O cantor","O cozinheiro","O vizinho","O pai","O filho","O irmão","O avô","O tio","O enfermeiro","O motorista","O secretário","O escritor","O ator"]
TEMPLATES = ["{subj} é muito", "{subj} parece", "{subj} ficou muito"]

random.seed(42)
GENDER_BENCHMARK = []
for adj_m, adj_f in ADJECTIVES:
    if adj_m == adj_f: continue
    for template in TEMPLATES:
        subj_f = random.choice(SUBJECTS_FEM)
        subj_m = random.choice(SUBJECTS_MASC)
        GENDER_BENCHMARK.append((template.format(subj=subj_f), adj_f, adj_m, "F"))
        GENDER_BENCHMARK.append((template.format(subj=subj_m), adj_m, adj_f, "M"))
random.shuffle(GENDER_BENCHMARK)
print(f"Benchmark: {len(GENDER_BENCHMARK)} items")

## 5. Evaluation functions

In [ ]:
from tqdm.auto import tqdm
from scipy.stats import fisher_exact

def get_token_prob(probs, word):
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    ids_ns = tokenizer.encode(word, add_special_tokens=False)
    all_ids = list(set(ids + ids_ns))
    return sum(probs[tid].item() for tid in all_ids)

def evaluate_benchmark(feature_ids=None, desc=""):
    results = []
    with ablate_multi(feature_ids or []):
        for ctx, correct, incorrect, gender in tqdm(GENDER_BENCHMARK, desc=desc, leave=False):
            ids = tokenizer.encode(ctx, return_tensors="pt").to(device)
            with torch.no_grad():
                logits = model(ids).logits
            probs = torch.softmax(logits[0, -1, :].float(), dim=-1)
            pc = get_token_prob(probs, correct)
            pi = get_token_prob(probs, incorrect)
            results.append({"gender": gender, "is_correct": pc > pi,
                            "p_correct": pc, "p_incorrect": pi})
    return results

def stats(results, baseline=None):
    fem = [r for r in results if r["gender"] == "F"]
    masc = [r for r in results if r["gender"] == "M"]
    n_correct = sum(1 for r in results if r["is_correct"])
    fem_correct = sum(1 for r in fem if r["is_correct"])
    masc_correct = sum(1 for r in masc if r["is_correct"])
    s = {"overall": n_correct/len(results),
         "feminine": fem_correct/len(fem),
         "masculine": masc_correct/len(masc),
         "fem_correct": fem_correct, "masc_correct": masc_correct,
         "n_fem": len(fem), "n_masc": len(masc)}
    if baseline is not None:
        b_fem = baseline["fem_correct"]; b_masc = baseline["masc_correct"]
        fem_table = [[fem_correct, len(fem)-fem_correct], [b_fem, len(fem)-b_fem]]
        masc_table = [[masc_correct, len(masc)-masc_correct], [b_masc, len(masc)-b_masc]]
        _, s["fisher_p_fem"] = fisher_exact(fem_table)
        _, s["fisher_p_masc"] = fisher_exact(masc_table)
        s["fem_drop_pp"] = 100*(baseline["feminine"] - s["feminine"])
        s["masc_drop_pp"] = 100*(baseline["masculine"] - s["masculine"])
    return s

## 6. Sweep K = {0, 1, 2, 3, 5, 10, 20, 50}

In [ ]:
sweep = {}
baseline_stats = None

for k in K_VALUES:
    fids = TOP_FEATURE_IDS[:k] if k > 0 else []
    label = "baseline" if k == 0 else f"top-{k}"
    print(f"\n[K={k:>2}] features={fids[:8]}{'...' if k>8 else ''}")
    res = evaluate_benchmark(feature_ids=fids, desc=f"K={k}")
    s = stats(res, baseline=baseline_stats)
    if k == 0:
        baseline_stats = s
    sweep[k] = {"feature_ids": fids, "stats": s}
    if k == 0:
        print(f"   overall={s['overall']:.4f}  F={s['feminine']:.4f}  M={s['masculine']:.4f}")
    else:
        print(f"   overall={s['overall']:.4f}  F={s['feminine']:.4f}  M={s['masculine']:.4f}")
        print(f"   Δfem={s['fem_drop_pp']:+.2f}pp  Δmasc={s['masc_drop_pp']:+.2f}pp  pF={s['fisher_p_fem']:.4f}  pM={s['fisher_p_masc']:.4f}")

## 7. Tabela resumida

In [ ]:
print(f"\n{'K':>4}  {'overall':>9}  {'fem':>8}  {'masc':>8}  {'Δfem':>8}  {'Δmasc':>8}  {'pF':>9}  {'pM':>9}")
print("-" * 80)
for k in K_VALUES:
    s = sweep[k]["stats"]
    if k == 0:
        print(f"{k:>4}  {s['overall']:>9.4f}  {s['feminine']:>8.4f}  {s['masculine']:>8.4f}  {'-':>8}  {'-':>8}  {'-':>9}  {'-':>9}")
    else:
        print(f"{k:>4}  {s['overall']:>9.4f}  {s['feminine']:>8.4f}  {s['masculine']:>8.4f}  {s['fem_drop_pp']:>+8.2f}  {s['masc_drop_pp']:>+8.2f}  {s['fisher_p_fem']:>9.4f}  {s['fisher_p_masc']:>9.4f}")

## 8. Plot K vs degradation

In [ ]:
import matplotlib.pyplot as plt

ks = K_VALUES
fems = [sweep[k]["stats"]["feminine"] for k in ks]
mascs = [sweep[k]["stats"]["masculine"] for k in ks]
ovrs = [sweep[k]["stats"]["overall"] for k in ks]

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(ks, ovrs, "o-", label="overall", linewidth=2)
ax.plot(ks, fems, "s--", label="feminine", alpha=0.8)
ax.plot(ks, mascs, "^--", label="masculine", alpha=0.8)
ax.set_xlabel("K = #features ablated (top-K por |effect size| de gender)")
ax.set_ylabel("benchmark accuracy")
ax.set_title(f"Gemma 2 9B — L{LAYER} — multi-feature ablation")
ax.set_xscale("symlog", linthresh=1)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig("gemma9b_multifeature_ablation_L20.png", dpi=150)
plt.show()
print("Plot saved to gemma9b_multifeature_ablation_L20.png")

## 9. Salvar JSON

In [ ]:
out = {
    "experiment": "gemma9b_multifeature_ablation",
    "model": MODEL_NAME,
    "sae_release": SAE_RELEASE,
    "layer": LAYER,
    "K_values": K_VALUES,
    "top_feature_ids": TOP_FEATURE_IDS[:max(K_VALUES)],
    "sweep": {str(k): sweep[k] for k in K_VALUES},
    "benchmark_n": len(GENDER_BENCHMARK),
}
with open("gemma9b_multifeature_ablation_L20.json", "w") as f:
    json.dump(out, f, indent=2, ensure_ascii=False)
print("Saved to gemma9b_multifeature_ablation_L20.json")

## 10. Interpretation

**Expected patterns:**
- **Gradual degradation (Δfem grows with K, p_fem drops with K):** confirms redundancy — the gender signal is distributed and needs several features to be broken.
- **Early saturated degradation (large Δfem already at K=2 or K=3):** the signal is in few features but B.2.3 missed it — the probes' top-1 alone is not sufficient.
- **No degradation until high K:** very strong redundancy; the model recovers the signal even with 50 ablated features (the most robust scenario for the paper's thesis).

**The result goes into the paper as a cross-scale table:**

| Layer | Model | K=1 Δfem | K=3 Δfem | K=5 Δfem | K=10 Δfem |
|---|---|---|---|---|---|
| L17 | 2B | −20.18pp** | — | — | — |
| L20 | 9B | (from B.2.3) | (this exp) | (this exp) | (this exp) |